In [1]:
import pandas as pd

# 1. Load Titanic Dataset (Classification)
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_titanic = pd.read_csv(titanic_url)
print("Titanic dataset loaded successfully!")
print(f"Rows: {df_titanic.shape[0]}, Columns: {df_titanic.shape[1]}\n")

# 2. Load House Prices Dataset (Regression)
# This is the exact train.csv from the Kaggle competition, hosted on a public repo
house_prices_url = "https://raw.githubusercontent.com/eddiexunyc/DATA605_FINAL/main/Resources/train.csv"
df_house = pd.read_csv(house_prices_url)
print("House Prices dataset loaded successfully!")
print(f"Rows: {df_house.shape[0]}, Columns: {df_house.shape[1]}")

Titanic dataset loaded successfully!
Rows: 891, Columns: 12

House Prices dataset loaded successfully!
Rows: 1460, Columns: 81


In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

print("--- PART 1: TITANIC CLASSIFICATION ---")

# 1. Load Data
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_titanic = pd.read_csv(titanic_url)

# 2. Preprocessing
# Keep it simple: use Pclass, Sex, Age, and Fare
df_titanic = df_titanic[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].copy()

# Fill missing Ages with the median
df_titanic['Age'] = df_titanic['Age'].fillna(df_titanic['Age'].median())

# Convert 'Sex' (male/female) to numbers (1/0)
df_titanic['Sex'] = LabelEncoder().fit_transform(df_titanic['Sex'])

# Define Features (X) and Target (y)
X_clf = df_titanic.drop('Survived', axis=1)
y_clf = df_titanic['Survived']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# 3. Initialize Models
xgb_clf = XGBClassifier(eval_metric='logloss', random_state=42)
lgb_clf = LGBMClassifier(random_state=42, verbose=-1)

# 4. Cross-Validation (Scikit-Learn)
xgb_cv_scores = cross_val_score(xgb_clf, X_train_c, y_train_c, cv=5, scoring='accuracy')
print(f"XGBoost 5-Fold CV Accuracy: {xgb_cv_scores.mean():.4f}")

lgb_cv_scores = cross_val_score(lgb_clf, X_train_c, y_train_c, cv=5, scoring='accuracy')
print(f"LightGBM 5-Fold CV Accuracy: {lgb_cv_scores.mean():.4f}\n")

# 5. Final Training & Metrics
xgb_clf.fit(X_train_c, y_train_c)
y_pred_c = xgb_clf.predict(X_test_c)

print("XGBoost Final Test Classification Report:")
print(classification_report(y_test_c, y_pred_c))
print("-" * 40)

--- PART 1: TITANIC CLASSIFICATION ---
XGBoost 5-Fold CV Accuracy: 0.8034
LightGBM 5-Fold CV Accuracy: 0.8160

XGBoost Final Test Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.84      0.83       105
           1       0.77      0.76      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.80      0.80      0.80       179

----------------------------------------


In [3]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("\n--- PART 2: HOUSE PRICE REGRESSION ---")

# 1. Load Data
house_prices_url = "https://raw.githubusercontent.com/eddiexunyc/DATA605_FINAL/main/Resources/train.csv"
df_house = pd.read_csv(house_prices_url)

# 2. Preprocessing
# Select only numerical columns for a streamlined model
num_cols = df_house.select_dtypes(include=[np.number]).columns
df_house = df_house[num_cols].copy()

# Drop the 'Id' column as it has no predictive value
if 'Id' in df_house.columns:
    df_house = df_house.drop('Id', axis=1)

# Fill missing numerical values with the median of their respective columns
df_house = df_house.fillna(df_house.median())

# Define Features (X) and Target (y)
X_reg = df_house.drop('SalePrice', axis=1)
y_reg = df_house['SalePrice']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 3. Initialize Models
xgb_reg = XGBRegressor(random_state=42)
lgb_reg = LGBMRegressor(random_state=42, verbose=-1)

# 4. Cross-Validation (Scikit-Learn)
# Using negative root mean squared error (common for regression CV)
xgb_cv_rmse = cross_val_score(xgb_reg, X_train_r, y_train_r, cv=5, scoring='neg_root_mean_squared_error')
print(f"XGBoost 5-Fold CV RMSE: {-xgb_cv_rmse.mean():.2f}")

lgb_cv_rmse = cross_val_score(lgb_reg, X_train_r, y_train_r, cv=5, scoring='neg_root_mean_squared_error')
print(f"LightGBM 5-Fold CV RMSE: {-lgb_cv_rmse.mean():.2f}\n")

# 5. Final Training & Metrics
xgb_reg.fit(X_train_r, y_train_r)
y_pred_r = xgb_reg.predict(X_test_r)

print("XGBoost Final Test Metrics:")
print(f"RMSE (Root Mean Squared Error): {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.2f}")
print(f"R-squared: {r2_score(y_test_r, y_pred_r):.4f}")


--- PART 2: HOUSE PRICE REGRESSION ---
XGBoost 5-Fold CV RMSE: 31931.55
LightGBM 5-Fold CV RMSE: 29920.99

XGBoost Final Test Metrics:
RMSE (Root Mean Squared Error): 30092.99
R-squared: 0.8819
